In [29]:
import subprocess, time, requests
import os
# os.environ["VLLM_ATTENTION_BACKEND"] = "XFORMERS"

process = subprocess.Popen(
    ["python", "-m", "vllm.entrypoints.openai.api_server",
     "--model", "Qwen/Qwen2.5-3B-Instruct",
     "--port", "8000",
     "--max-model-len", "4096",
     "--gpu-memory-utilization", "0.9",
     "--dtype", "half",
    #  "--enforce-eager"],           # disables CUDA graphs + uses eager mode, avoids FA2
    #  "--attention-backend", "xformers"
     ],
    stdout=open("vllm.log", "w"),
    stderr=subprocess.STDOUT
)

for i in range(2):
    try:
        r = requests.get("http://localhost:8000/v1/models")
        if r.status_code == 200:
            print(f"Server ready after {(i+1)*5}s!")
            break
    except:
        pass
    time.sleep(5)
else:
    print("Server didn't start. Checking logs:")


Server didn't start. Checking logs:


In [25]:
from openai import OpenAI
client = OpenAI(base_url="http://localhost:8000/v1", api_key="dummy")
resp = client.chat.completions.create(
    model="Qwen/Qwen2.5-3B-Instruct",
    messages=[{"role": "user", "content": "What is 2+2?"}],
    max_tokens=50
)
print(resp.choices[0].message.content)


2 + 2 equals 4.


In [19]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.makedirs("results", exist_ok=True)

DATASET = 'wikitq'
DATASPLIT = "test"
G_TYPE = "pot"
N_SHOTS = 8
N_SAMPLE = 1
TEMPERATURE = 0.0
ENGINE = 'Qwen/Qwen2.5-3B-Instruct'
API_FILE = "key.txt"
LOG_FILE = f"results/2_{G_TYPE}_{DATASET}_{DATASPLIT}_{N_SHOTS}_{N_SAMPLE}_{ENGINE.replace('/', '_')}_{TEMPERATURE}.log"

os.system(f"""python scripts/execute_pot.py --dataset {DATASET} \
--dataset_split {DATASPLIT} \
--prompt_file templates/prompts/wikitq_pot_ic.txt \
--output_program_file 2_{G_TYPE}_{DATASET}.json \
--n_parallel_prompts 1 \
--n_processes 1 \
--n_shots {N_SHOTS} \
--max_generation_tokens 256 \
--max_api_total_tokens 4096 \
--engine {ENGINE} \
--api_config_file {API_FILE} 2>&1 | tee {LOG_FILE}""")

/home/kevin/miniconda3/envs/augment/lib/python3.11/site-packages/fuzzywuzzy/fuzz.py:11: UserWarning: Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning
  warnings.warn('Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning')
Annotating Args info:
dataset: wikitq
dataset_split: test
api_config_file: key.txt
prompt_file: templates/prompts/wikitq_pot_ic.txt
system_prompt_file: templates/prompts/default_system.txt
output_program_file: 2_pot_wikitq.json
save_dir: results/
n_processes: 1
prompt_style: create_table_select_3_full_table
generate_type: nsql
n_shots: 8
seed: 42
engine: Qwen/Qwen2.5-3B-Instruct
n_parallel_prompts: 1
max_generation_tokens: 256
max_api_total_tokens: 4096
temperature: 0.4
sampling_n: 20
top_p: 1.0
stop_tokens: ['\n\n']
verbose: False
debug: False
max_sample_num: 99999999999999
==================== Prompt file: /home/kevin/projects/cse517/Augment_QA/scripts/../templates/prompts/wikitq_pot_ic.tx

Aborted (core dumped)


0

In [13]:
os.system(f"""python scripts/execute_pot_finqa.py \
    --engine Qwen/Qwen2.5-3B-Instruct \
    --dataset finqa \
    --dataset_split test \
    --max_generation_tokens 256 \
    --max_api_total_tokens 4001 \
    --n_processes 1 \
    --output_program_file pot_finqa_test_qwen3b.json""")

/home/kevin/miniconda3/envs/augment/lib/python3.11/site-packages/fuzzywuzzy/fuzz.py:11: UserWarning: Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning
  warnings.warn('Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning')


POT FinQA Args:
  dataset: finqa
  dataset_split: test
  api_config_file: key.txt
  output_program_file: pot_finqa_test_qwen3b.json
  save_dir: results/
  engine: Qwen/Qwen2.5-3B-Instruct
  n_processes: 1
  max_generation_tokens: 256
  max_api_total_tokens: 4001
  max_sample_num: 99999999
  sampling_n: 1
  verbose: False
  debug: False

******* Running POT on FinQA (158 examples) *******


POT-FinQA Worker 0:  84%|████████▎ | 132/158 [00:42<00:08,  2.97it/s, acc=17/132 (12.88%)]POT-FinQA Worker 0:  84%|████████▎ | 132/158 [1:21:59<16:09, 37.27s/it, acc=17/132 (12.88%)]
Traceback (most recent call last):
  File "/home/kevin/projects/cse517/Augment_QA/scripts/execute_pot_finqa.py", line 342, in <module>
    main()
  File "/home/kevin/projects/cse517/Augment_QA/scripts/execute_pot_finqa.py", line 291, in main
    res = worker_annotate(0, args, generate_eids_group[0], dataset, tokenizer)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/kevin/projects/cse517/Augment_QA/scripts/execute_pot_finqa.py", line 203, in worker_annotate
    while len(tokenizer.tokenize(query_table)) >= max_prompt_tokens - 2000:
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/kevin/miniconda3/envs/augment/lib/python3.11/site-packages/transformers/tokenization_utils_fast.py", line 435, in tokenize
    return self.encode_plus(text=text, text_pair=pa

2

In [12]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_OFFLINE"] = "1"
os.makedirs("results", exist_ok=True)

DATASET = "wikitq"
DATASPLIT = "test"
G_TYPE = "nsql"
N_SHOTS = 8
N_SAMPLE = 1
TEMPERATURE = 0.0
ENGINE = "Qwen/Qwen2.5-3B-Instruct"
API_FILE = "key.txt"

PROG_FILE = f"{G_TYPE}_{DATASET}_{DATASPLIT}_{N_SHOTS}_{N_SAMPLE}_{ENGINE.replace('/', '_')}_{TEMPERATURE}.json"
EXEC_FILE = f"{G_TYPE}_{DATASET}_{DATASPLIT}_{N_SHOTS}_{N_SAMPLE}_{ENGINE.replace('/', '_')}_{TEMPERATURE}_exec.json"
LOG_FILE = f"results/{G_TYPE}_{DATASET}_{DATASPLIT}_{ENGINE.replace('/', '_')}.log"

# Step 1: Generate binder programs
# os.system(f"""python scripts/annotate_binder_program.py --dataset {DATASET} \
# --dataset_split {DATASPLIT} \
# --prompt_file templates/prompts/wikitq_nsqlcot_ic.txt \
# --system_prompt_file templates/prompts/wikitq_nsqlcot_system.txt \
# --output_program_file {PROG_FILE} \
# --n_parallel_prompts 1 \
# --n_processes 1 \
# --n_shots {N_SHOTS} \
# --max_generation_tokens 256 \
# --max_api_total_tokens 4096 \
# --temperature {TEMPERATURE} \
# --sampling_n {N_SAMPLE} \
# --engine {ENGINE} \
# --api_config_file {API_FILE} 2>&1 | tee {LOG_FILE}""")

# Step 2: Execute binder programs
os.system(f"""python scripts/execute_binder_program.py --dataset {DATASET} \
--dataset_split {DATASPLIT} \
--qa_retrieve_pool_file templates/qa_retrieve_pool/qa_retrieve_pool.json \
--input_program_file {PROG_FILE} \
--output_program_execution_file {EXEC_FILE} \
--vote_method count \
--engine {ENGINE} \
--n_processes 1 \
--api_config_file {API_FILE} \
--use_cot 2>&1 | tee -a {LOG_FILE}""")


/home/kevin/miniconda3/envs/augment/lib/python3.11/site-packages/fuzzywuzzy/fuzz.py:11: UserWarning: Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning
  warnings.warn('Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning')
Args info:
dataset: wikitq
dataset_split: test
api_config_file: key.txt
save_dir: results/
qa_retrieve_pool_file: templates/qa_retrieve_pool/qa_retrieve_pool.json
input_program_file: nsql_wikitq_test_8_1_Qwen_Qwen2.5-3B-Instruct_0.0.json
output_program_execution_file: nsql_wikitq_test_8_1_Qwen_Qwen2.5-3B-Instruct_0.0_exec.json
n_processes: 1
use_majority_vote: False
allow_none_and_empty_answer: False
allow_error_answer: False
answer_placeholder: <error|empty>
vote_method: count
answer_biased: None
answer_biased_weight: None
process_program_with_fuzzy_match_on_db: False
use_cot: True
engine: Qwen/Qwen2.5-3B-Instruct
prompt_style: create_table_select_3_full_table
seed: 42
verbose: False
debug:

0

In [28]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_OFFLINE"] = "1"
os.makedirs("results", exist_ok=True)

os.system(f"""python scripts/execute_pot_tatqa.py \
    --engine Qwen/Qwen2.5-3B-Instruct \
    --dataset tatqa \
    --dataset_split validation \
    --max_generation_tokens 256 \
    --max_api_total_tokens 4096 \
    --n_processes 1 \
    --output_program_file pot_tatqa_validation_qwen3b.json 2>&1 | tee results/pot_tatqa_validation_qwen3b.log""")

POT TatQA Args:
  dataset: tatqa
  dataset_split: validation
  api_config_file: key.txt
  output_program_file: pot_tatqa_validation_qwen3b.json
  save_dir: results/
  engine: Qwen/Qwen2.5-3B-Instruct
  n_processes: 1
  max_generation_tokens: 256
  max_api_total_tokens: 4096
  max_sample_num: 99999999
  sampling_n: 1
  verbose: False
  debug: False

******* Running POT on TatQA (507 examples) *******
POT-TatQA Worker 0:   0%|          | 0/507 [00:00<?, ?it/s]
running... 0


bruh1
{'table': {'header': ['', '2019', '2018', '2017'], 'rows': [['Fixed Price', '$  1,452.4', '$  1,146.2', '$  1,036.9'], ['Other', '44.1', '56.7', '70.8'], ['Total sales', '$1,496.5', '$1,202.9', '$1,107.7']], 'page_title': '3ffd9053-a45d-491c-957a-1b2fa0af0570'}, 'question': 'What is the amount of total sales in 2019?', 'id': '4960801d-277d-4f79-8eca-c4d0200fa9d6', 'answer_text': ['$1,496.5'], 'table_id': '3ffd9053-a45d-491c-957a-1b2fa0af0570', 'document_input': ['Sales by Contract Type: Substantially all of our

2